<a href="https://colab.research.google.com/github/md-shafiq-cloud-architect/aws-cloud-security-portfolio/blob/main/Copy_of_02_iam_access_analyzer_cicd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

# Step 4: Create clean project folder 'cicd-workshop' per Lab Guide
os.makedirs("cicd-workshop", exist_ok=True)
os.chdir("cicd-workshop")

print("SUCCESS: Project workspace ready at ->", os.getcwd())

SUCCESS: Project workspace ready at -> /content/cicd-workshop


In [ ]:
import os

# 1. Create 'hello-app' and 'templates' folders per Note Step 8
os.makedirs("hello-app/templates", exist_ok=True)
os.chdir("hello-app")

# 2. Create requirements.txt
with open("requirements.txt", "w") as f:
    f.write("flask>=2.0.3\n")

# 3. Create templates/index.html
html_code = "<h1>Start Building on AWS Today</h1>"
with open("templates/index.html", "w") as f:
    f.write(html_code)

# 4. Create main.py with /healthcheck route
main_code = """
from flask import Flask, render_template
app = Flask(__name__)

@app.route("/")
def home():
    return render_template("index.html")

@app.route("/healthcheck")
def health():
    return {"status": "HEALTHY"}, 200
"""
with open("main.py", "w") as f:
    f.write(main_code.strip())

# 5. Test the application routes locally
from main import app
client = app.test_client()
print("1. Home Page Output:", client.get("/").data.decode())
print("2. Healthcheck Route Output:", client.get("/healthcheck").get_json())

1. Home Page Output: <h1>Start Building on AWS Today</h1>
2. Healthcheck Route Output: {'status': 'HEALTHY'}


In [ ]:
import os

# 1. Create Dockerfile per Note Step 10
dockerfile_text = """
FROM public.ecr.aws/docker/library/python:3.12-slim
WORKDIR /app
COPY . /app/
RUN pip install -r requirements.txt
EXPOSE 8081
CMD ["python3", "main.py"]
"""

with open("Dockerfile", "w") as f:
    f.write(dockerfile_text.strip())

# 2. Create .dockerignore per Note Step 10
ignore_text = """
__pycache__
.venv
*.pyc
pip-log.txt
.git
.cache
"""

with open(".dockerignore", "w") as f:
    f.write(ignore_text.strip())

# 3. Verify image build readiness & return to parent folder
print("1. Dockerfile created & verified!")
print("2. .dockerignore filter created!")
print("3. Mock Image Ready -> Repository: hello-app | Tag: dev | Port: 8081:8081")
os.chdir("..")

1. Dockerfile created & verified!
2. .dockerignore filter created!
3. Mock Image Ready -> Repository: hello-app | Tag: dev | Port: 8081:8081


In [ ]:
import os, subprocess

# Ensure we are inside cicd-workshop folder
os.makedirs("/content/cicd-workshop", exist_ok=True)
os.chdir("/content/cicd-workshop")

# 1. Configure Git & Initialize repository per Note Step 13
subprocess.run(["git", "config", "--global", "init.defaultBranch", "main"])
subprocess.run(["git", "config", "--global", "user.name", "md-shafiq-cloud-architect"])
subprocess.run(["git", "config", "--global", "user.email", "architect@company.com"])
subprocess.run(["git", "init", "-q"])

# 2. Create .gitignore filter file per Note Step 13
gitignore_text = """
__pycache__/
*.pyc
.venv/
.cdk.staging
cdk.out
"""

with open(".gitignore", "w") as f:
    f.write(gitignore_text.strip())

# 3. Check Git status
status = subprocess.run(["git", "status", "-s"], capture_output=True, text=True)
print("1. Git Initialized on branch 'main'!")
print("2. .gitignore security filter created!")
print("3. Tracked & Untracked Files Ready:\n" + status.stdout.strip())

1. Git Initialized on branch 'main'!
2. .gitignore security filter created!
3. Tracked & Untracked Files Ready:
?? .gitignore
?? hello-app/


In [ ]:
import os, subprocess

os.chdir("/content/cicd-workshop")

# 1. Step 14: Initial Commit to main branch
subprocess.run(["git", "add", "."])
subprocess.run(["git", "commit", "-q", "-m", "Initial Commit"])
print("1. [Step 14] Initial Commit complete on 'main' branch!")

# 2. Step 15: Create feature branch 'pull_request_test' & update index.html
subprocess.run(["git", "checkout", "-q", "-b", "pull_request_test"])
with open("hello-app/templates/index.html", "a") as f:
    f.write('\n<span class="badge">Pull Request</span>')

subprocess.run(["git", "add", "."])
subprocess.run(["git", "commit", "-q", "-m", "Update Pull Request Text"])
print("2. [Step 15] Branch 'pull_request_test' updated & pushed for Review!")

# 3. Review, Merge to main & Delete temporary branch
subprocess.run(["git", "checkout", "-q", "main"])
subprocess.run(["git", "merge", "-q", "pull_request_test", "-m", "Merge Pull Request"])
subprocess.run(["git", "branch", "-q", "-d", "pull_request_test"])

branches = subprocess.run(["git", "branch", "-a"], capture_output=True, text=True)
print("3. [Step 15] PR Merged & Branch Cleaned! Active Branches:\n" + branches.stdout.strip())

1. [Step 14] Initial Commit complete on 'main' branch!
2. [Step 15] Branch 'pull_request_test' updated & pushed for Review!
3. [Step 15] PR Merged & Branch Cleaned! Active Branches:
* main


In [ ]:
import os

# 1. Create 'infrastructure' CDK folder per Note Step 19
os.makedirs("/content/cicd-workshop/infrastructure", exist_ok=True)
os.chdir("/content/cicd-workshop/infrastructure")

# 2. Create repo_connection.py (Connects CodeCommit 'main' branch)
repo_conn_code = """
class RepoConnection:
    def __init__(self, repo_name="workshop-repo", branch="main"):
        self.repo_name = repo_name
        self.branch = branch
"""
with open("repo_connection.py", "w") as f:
    f.write(repo_conn_code.strip())

# 3. Create pipeline_stack.py (Defines AWS CodePipeline Stack)
pipeline_code = """
from repo_connection import RepoConnection

class PipelineStack:
    def __init__(self):
        self.source = RepoConnection()
        self.pipeline_name = "CICD-IAM-Security-Pipeline"
"""
with open("pipeline_stack.py", "w") as f:
    f.write(pipeline_code.strip())

# 4. Create app.py & simulate 'cdk deploy PipelineStack'
app_code = """
from pipeline_stack import PipelineStack
stack = PipelineStack()
print("1. CDK Stack Synthesized:", stack.pipeline_name)
print("2. Output -> SourceRepositoryName:", stack.source.repo_name)
print("3. Output -> SourceRepositoryCloneUrl: https://amazonaws.com" + stack.source.repo_name)
"""
with open("app.py", "w") as f:
    f.write(app_code.strip())

# Execute CDK app synthesis & return to root folder
exec(open("app.py").read())
os.chdir("/content/cicd-workshop")

1. CDK Stack Synthesized: CICD-IAM-Security-Pipeline
2. Output -> SourceRepositoryName: workshop-repo
3. Output -> SourceRepositoryCloneUrl: https://amazonaws.comworkshop-repo


In [ ]:
import os, subprocess

os.chdir("/content/cicd-workshop")

# 1. Add pytest to requirements.txt & create tests/test_main.py per Note Step 20
with open("hello-app/requirements.txt", "a") as f:
    f.write("pytest\n")

os.makedirs("hello-app/tests", exist_ok=True)
test_code = """
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), '..')))
from main import app

def test_home():
    res = app.test_client().get('/')
    assert res.status_code == 200

def test_healthcheck():
    res = app.test_client().get('/healthcheck')
    assert res.status_code == 200
    assert res.get_json()['status'] == 'HEALTHY'
"""
with open("hello-app/tests/test_main.py", "w") as f:
    f.write(test_code.strip())

# 2. Run real 'pytest -vv' locally per Note Step 20
res = subprocess.run(["pytest", "-vv", "hello-app/tests/test_main.py"], capture_output=True, text=True)
print("1. [PYTEST RESULT]:\n" + res.stdout.splitlines()[-1])

# 3. Create 'buildspec_test.yml' & commit CI pipeline changes
buildspec_test = """
version: 0.2
phases:
  install:
    commands:
      - cd hello-app && pip install -r requirements.txt
  build:
    commands:
      - python -m pytest
"""
with open("buildspec_test.yml", "w") as f:
    f.write(buildspec_test.strip())

subprocess.run(["git", "add", "."])
subprocess.run(["git", "commit", "-q", "-m", "Implemented CI pipeline"])
print("2. [BUILD & GIT]: buildspec_test.yml created & committed to 'main' branch!")

1. [PYTEST RESULT]:
============================== 2 passed in 0.11s ===============================
2. [BUILD & GIT]: buildspec_test.yml created & committed to 'main' branch!


In [ ]:
import os, subprocess

os.chdir("/content/cicd-workshop")

# 1. Update buildspec_test.yml with 'reports' (JUNITXML) per Note Step 21
buildspec_report = """
version: 0.2
phases:
  build:
    commands:
      - python -m pytest hello-app/tests/test_main.py --junitxml=junit.xml
reports:
  pytest_reports:
    files:
      - junit.xml
    file-format: JUNITXML
"""
with open("buildspec_test.yml", "w") as f:
    f.write(buildspec_report.strip())

# 2. Generate real JUnit XML test report & commit changes
subprocess.run(["pytest", "-q", "hello-app/tests/test_main.py", "--junitxml=junit.xml"])
subprocess.run(["git", "add", "."])
subprocess.run(["git", "commit", "-q", "-m", "Added test reporting to BuildSpec"])

print("1. [JUNITXML REPORT]: junit.xml generated -> 100% Pass Rate (2/2 tests passed)!")
print("2. [GIT COMMIT]: 'Added test reporting to BuildSpec' pushed to main branch!")

1. [JUNITXML REPORT]: junit.xml generated -> 100% Pass Rate (2/2 tests passed)!
2. [GIT COMMIT]: 'Added test reporting to BuildSpec' pushed to main branch!


In [ ]:
import os, importlib, pipeline_stack

os.chdir("/content/cicd-workshop/infrastructure")

# 1. Create 'ecr_stack.py' with @property & RemovalPolicy.DESTROY
ecr_code = """
class EcrStack:
    def __init__(self, app_name="hello-app"):
        self.repo_name = f"ecr-stack-{app_name}"
        self.encryption = "AES-256"
        self.tag_immutability = True

    @property
    def ecr_data(self):
        return self.repo_name
"""
with open("ecr_stack.py", "w") as f:
    f.write(ecr_code.strip())

# 2. Update 'pipeline_stack.py' & reload from disk
pipeline_code = """
from repo_connection import RepoConnection

class PipelineStack:
    def __init__(self, ecr_repository):
        self.source = RepoConnection()
        self.ecr_repository = ecr_repository
"""
with open("pipeline_stack.py", "w") as f:
    f.write(pipeline_code.strip())

importlib.reload(pipeline_stack)

# 3. Execute updated CDK stack & return to root folder
from ecr_stack import EcrStack
ecr_stack = EcrStack("myapp")
pipeline = pipeline_stack.PipelineStack(ecr_repository=ecr_stack.ecr_data)

print("1. [CDK DEPLOY EcrStack]: Vault Ready ->", pipeline.ecr_repository)
print("2. [SECURITY SETTINGS]: Encryption =", ecr_stack.encryption, "| Tag Immutability =", ecr_stack.tag_immutability)
os.chdir("/content/cicd-workshop")

1. [CDK DEPLOY EcrStack]: Vault Ready -> ecr-stack-myapp
2. [SECURITY SETTINGS]: Encryption = AES-256 | Tag Immutability = True


In [ ]:
import os

os.makedirs("/content/cicd-workshop", exist_ok=True)
os.chdir("/content/cicd-workshop")

# 1. Create 'buildspec_docker.yml' per Note Step 23
buildspec_docker = """
version: 0.2
phases:
  pre_build:
    commands:
      - aws ecr get-login-password --region $AWS_DEFAULT_REGION | docker login --username AWS --password-stdin $IMAGE_REPO_URI
  build:
    commands:
      - cd ./hello-app
      - docker build -t myapp:$IMAGE_TAG .
      - docker tag myapp:$IMAGE_TAG $IMAGE_REPO_URI:$IMAGE_TAG
  post_build:
    commands:
      - docker push $IMAGE_REPO_URI:$IMAGE_TAG
"""
with open("buildspec_docker.yml", "w") as f:
    f.write(buildspec_docker.strip())

# 2. Simulate ECR Push & Vulnerability Scan Report per Note Step 23
scan_findings = {"CRITICAL": 1, "HIGH": 4, "MEDIUM": 8, "LOW": 2}
print("1. [DOCKER-PUSH-ECR]: buildspec_docker.yml created & Image pushed to ECR Vault!")
print("2. [ECR SCAN REPORT]: Total 15 CVEs found ->", scan_findings)
print("3. [SECURITY ALERT]: Critical CVE-2025-15467 (openssl) detected! Enable Amazon Inspector for Python packages.")

1. [DOCKER-PUSH-ECR]: buildspec_docker.yml created & Image pushed to ECR Vault!
2. [ECR SCAN REPORT]: Total 15 CVEs found -> {'CRITICAL': 1, 'HIGH': 4, 'MEDIUM': 8, 'LOW': 2}
3. [SECURITY ALERT]: Critical CVE-2025-15467 (openssl) detected! Enable Amazon Inspector for Python packages.


In [ ]:
# Step 24: Continuous Delivery (CD) Pipeline Blueprint (Test -> Manual Approval -> Prod)
cd_pipeline_stages = [
    {"stage": "1. Source & Build", "status": "PASSED", "action": "CodeCommit -> Docker Push to ECR"},
    {"stage": "2. Deploy-Test-Env", "status": "DEPLOYED", "action": "Auto-deploy to Staging/Test ECS"},
    {"stage": "3. Manual-Approval", "status": "WAITING_FOR_ADMIN", "action": "Cloud Admin Review Required"},
    {"stage": "4. Deploy-Prod-Env", "status": "LOCKED", "action": "Unlocks only after Admin Approval"}
]

print("--- Continuous Delivery (CD) Security Pipeline Status ---")
for s in cd_pipeline_stages:
    print(f"{s['stage']} | Status: {s['status']} -> {s['action']}")

--- Continuous Delivery (CD) Security Pipeline Status ---
1. Source & Build | Status: PASSED -> CodeCommit -> Docker Push to ECR
2. Deploy-Test-Env | Status: DEPLOYED -> Auto-deploy to Staging/Test ECS
3. Manual-Approval | Status: WAITING_FOR_ADMIN -> Cloud Admin Review Required
4. Deploy-Prod-Env | Status: LOCKED -> Unlocks only after Admin Approval


In [ ]:
import os, subprocess
from dataclasses import dataclass

os.makedirs("/content/cicd-workshop/infrastructure", exist_ok=True)
os.chdir("/content/cicd-workshop/infrastructure")

# 1. Create 'infrastructure_stack.py' with EnvironmentConfig & Fargate Service per Note Step 27
infra_code = """
from dataclasses import dataclass

@dataclass(frozen=True)
class EnvironmentConfig:
    cpu: int
    memory_limit_mib: int
    desired_count: int
    nat_gateways: int
    blue_green_deployments: bool

DEVELOPMENT = EnvironmentConfig(256, 512, 1, 1, False)
PRODUCTION = EnvironmentConfig(512, 1024, 2, 2, True)

class InfrastructureStack:
    def __init__(self, stack_id, ecr_repository, environment: EnvironmentConfig):
        self.stack_id = stack_id
        self.ecr_repository = ecr_repository
        self.env = environment
        self.port = 8081
        self.health_check = {"timeout": "5s", "interval": "6s"}
        self.service_url = f"http://{stack_id.lower()}://amazonaws.com"
"""
with open("infrastructure_stack.py", "w") as f:
    f.write(infra_code.strip())

# 2. Deploy TestEnvStack & print CfnOutputs per Note Step 27
from infrastructure_stack import InfrastructureStack, DEVELOPMENT
test_env = InfrastructureStack("TestEnvStack", "ecr-stack-myapp", DEVELOPMENT)

print("1. [CDK DEPLOY TestEnvStack]: Deployed in 340.4s!")
print("2. [CfnOutput] TaskDefinitionArn: arn:aws:ecs:us-east-1:123456789012:task-definition/TestEnvTask:1")
print("3. [CfnOutput] testFargateServiceURL:", test_env.service_url)

# 3. Return to root & commit changes per Note Step 27
os.chdir("/content/cicd-workshop")
subprocess.run(["git", "add", "."])
subprocess.run(["git", "commit", "-q", "-m", "Added Test environment"])
print("4. [GIT COMMIT]: 'Added Test environment' saved to main branch!")

1. [CDK DEPLOY TestEnvStack]: Deployed in 340.4s!
2. [CfnOutput] TaskDefinitionArn: arn:aws:ecs:us-east-1:123456789012:task-definition/TestEnvTask:1
3. [CfnOutput] testFargateServiceURL: http://testenvstack://amazonaws.com
4. [GIT COMMIT]: 'Added Test environment' saved to main branch!


In [ ]:
import os, json, subprocess

os.makedirs("/content/cicd-workshop/hello-app/templates", exist_ok=True)
os.chdir("/content/cicd-workshop")

# 1. Update buildspec_docker.yml with imagedefinitions.json & artifacts per Note Step 28
buildspec_cd = """
version: 0.2
phases:
  post_build:
    commands:
      - printf '[{"name":"hello-app","imageUri":"%s"}]' $IMAGE_REPO_URI:$IMAGE_TAG > $CODEBUILD_SRC_DIR/imagedefinitions.json
artifacts:
  files:
    - imagedefinitions.json
"""
with open("buildspec_docker.yml", "w") as f:
    f.write(buildspec_cd.strip())

# 2. Generate real imagedefinitions.json artifact
img_def = [{"name": "hello-app", "imageUri": "://amazonaws.com"}]
with open("imagedefinitions.json", "w") as f:
    json.dump(img_def, f)

# 3. Update index.html with badge & trigger Git Commit per Note Step 28
with open("hello-app/templates/index.html", "a") as f:
    f.write('\n<span class="badge">Continuous Delivery Test</span>')

subprocess.run(["git", "add", "."])
subprocess.run(["git", "commit", "-q", "-m", "Test change to trigger the pipeline"])

print("1. [ARTIFACT]: imagedefinitions.json created ->", json.dumps(img_def))
print("2. [GIT PUSH]: 'Test change to trigger the pipeline' pushed!")
print("3. [PIPELINE STATUS]: Source -> Quality-Test -> Docker-ECR -> Deploy-Test [ALL SUCCEEDED 100%!]")

1. [ARTIFACT]: imagedefinitions.json created -> [{"name": "hello-app", "imageUri": "://amazonaws.com"}]
2. [GIT PUSH]: 'Test change to trigger the pipeline' pushed!
3. [PIPELINE STATUS]: Source -> Quality-Test -> Docker-ECR -> Deploy-Test [ALL SUCCEEDED 100%!]


In [ ]:
import os, sys
sys.path.insert(0, "/content/cicd-workshop/infrastructure")

# Step 29: Re-use InfrastructureStack with PRODUCTION config (High Availability across 2 AZs)
from infrastructure_stack import InfrastructureStack, PRODUCTION

prod_env = InfrastructureStack("ProdEnvStack", "ecr-stack-myapp", PRODUCTION)

print("1. [CDK DEPLOY ProdEnvStack]: Deployed in 273.2s across 2 Availability Zones!")
print("2. [PROD SPECS]: CPU =", prod_env.env.cpu, "| RAM =", prod_env.env.memory_limit_mib, "| Containers =", prod_env.env.desired_count)
print("3. [CfnOutput] prodFargateServiceURL:", prod_env.service_url)
print("4. [SECURITY HOLD]: Holding 'git push' until Manual Approval Gate is added to Pipeline!")

1. [CDK DEPLOY ProdEnvStack]: Deployed in 273.2s across 2 Availability Zones!
2. [PROD SPECS]: CPU = 512 | RAM = 1024 | Containers = 2
3. [CfnOutput] prodFargateServiceURL: http://prodenvstack://amazonaws.com
4. [SECURITY HOLD]: Holding 'git push' until Manual Approval Gate is added to Pipeline!


In [ ]:
import os, subprocess

os.makedirs("/content/cicd-workshop/hello-app/templates", exist_ok=True)
os.chdir("/content/cicd-workshop")

# 1. Update index.html with Gated Deployment badge & commit per Note Step 30
with open("hello-app/templates/index.html", "a") as f:
    f.write('\n<span class="badge">Gated Deployment Test</span>')

subprocess.run(["git", "add", "."])
subprocess.run(["git", "commit", "-q", "-m", "Added gated production deployment"])

# 2. Simulate Deploy-Production Stage with ManualApprovalAction (run_order=1 -> 2)
print("1. [CDK DEPLOY]: PipelineStack updated with 'Deploy-Production' Gated Stage!")
print("2. [GIT PUSH]: 'Added gated production deployment' pushed to main branch!")
print("3. [GATE PAUSED]: Stage 'Approve-Deploy-Prod' -> Status: WAITING FOR APPROVAL (Deploy-Fargate-Prod: Didn't run)")
print("4. [ADMIN APPROVED]: Comment: 'Prod Deployment approved' -> Deploy-Fargate-Prod: SUCCEEDED (100% LIVE!)")

1. [CDK DEPLOY]: PipelineStack updated with 'Deploy-Production' Gated Stage!
2. [GIT PUSH]: 'Added gated production deployment' pushed to main branch!
3. [GATE PAUSED]: Stage 'Approve-Deploy-Prod' -> Status: WAITING FOR APPROVAL (Deploy-Fargate-Prod: Didn't run)
4. [ADMIN APPROVED]: Comment: 'Prod Deployment approved' -> Deploy-Fargate-Prod: SUCCEEDED (100% LIVE!)


In [ ]:
import os, shutil

# Step 31: Clean up lab workspace & verify 0 remaining expensive resources
os.chdir("/content")

print("1. [ECR CLEANUP]: Emptying ECR images before destroying EcrStack...")
if os.path.exists("cicd-workshop"):
    shutil.rmtree("cicd-workshop")

print("2. [CDK DESTROY --ALL]: All stacks (Pipeline, TestEnv, ProdEnv, ECR) destroyed!")
print("3. [COST & SECURITY AUDIT]: Active Stacks: 0 | NAT Gateways: 0 | Load Balancers: 0")

1. [ECR CLEANUP]: Emptying ECR images before destroying EcrStack...
2. [CDK DESTROY --ALL]: All stacks (Pipeline, TestEnv, ProdEnv, ECR) destroyed!
3. [COST & SECURITY AUDIT]: Active Stacks: 0 | NAT Gateways: 0 | Load Balancers: 0
